# Deep Dive Stellantis — Fastback & Pulse

Roda o pipeline de Deep Dive para os dois modelos TikTok da Stellantis (fastback e pulse),  
cada um com padrão de slug próprio (`product_level_3` / `product_level_3_pulse`) declarado  
em `data/vehicle_specs.yaml`, a partir do registry `configs/clients_registry.yaml`.

**Fluxo:**
1. Carregar registry  
2. Inspecionar canais Meridian (preencher `media_var` nos YAMLs antes de rodar em massa)  
3. Batch run (sequencial, com error handling por modelo)  
4. Meta-análise: shares, ROAS index, proxy_ratio por modelo × dimensão


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("../src"))

import warnings
import pandas as pd
import plotly.io as pio

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.3f}".format)

pio.renderers.default = "notebook"

REGISTRY_PATH   = "../configs/clients_registry.yaml"
OUTPUT_BASE_DIR = "../outputs/stellantis"

## 1. Registry

In [3]:
from batch import load_registry, _iter_runs

registry = load_registry(REGISTRY_PATH)

print(f"Clientes: {list(registry.keys())}\n")
for client_name, run_key, cfg in _iter_runs(registry):
    print(f"  {run_key:35s}  specs={cfg.get('specs_path','?')}")

Clientes: ['stellantis']

  stellantis_fastback                  specs=stellantis_fastback.yaml
  stellantis_pulse                     specs=stellantis_pulse.yaml


> **Antes de continuar:** preencha `upgrade_run_id`, `media_var`, `workspace_dd`, `start_date` e `end_date`  
> em `configs/stellantis_fastback.yaml` e `configs/stellantis_pulse.yaml`

## 3. Batch Run

In [ ]:
from batch import run_deep_dive_batch

# Filtros disponíveis:
#   clients=["stellantis"]              → fastback + pulse
#   clients=["stellantis_fastback"]     → só fastback
#   clients=["stellantis_pulse"]        → só pulse

all_results, all_diags, errors = run_deep_dive_batch(
    registry=registry,
    registry_path=REGISTRY_PATH,
    output_base_dir=OUTPUT_BASE_DIR,
    clients=["stellantis"],
    verbose=False,
)

print(f"\nSucesso: {list(all_results.keys())}")
if errors:
    print(f"\nErros: {list(errors.keys())}")
    for run_key, tb in errors.items():
        print(f"\n--- {run_key} ---")
        print(tb[-800:])

## 4. Resultados

In [ ]:
from batch import rollup_contribs_ts, apply_hierarchy_rollups
from plots import analyze_deepdive

for run_key, result in all_results.items():
    print(f"\n{'#'*20} {run_key} {'#'*20}")
    _ = analyze_deepdive(result)

    rollup_shares = apply_hierarchy_rollups(result)
    anchor = float(result.media_dd_contrib.sum())

    for dim in result.config.dims:
        c_df = result.contribs[dim]
        ts_rollups = rollup_contribs_ts(c_df, dim, result.config.vehicle_spec)
        for level, rollup_df in ts_rollups.items():
            total = float(rollup_df.sum().sum())
            shares_df = rollup_shares.get(dim, {}).get(level)
            spend_share_lookup = {}
            if shares_df is not None and not shares_df.empty:
                spend_share_lookup = dict(zip(shares_df["item"], shares_df["share_spend"]))

            print(f"\n{'='*66}")
            print(f"  {dim} → {level}  (total={total:,.0f}  {total/anchor:.1%} do âncora)")
            print(f"{'='*66}")
            print(f"  {'Item':<42s} {'Absoluto':>12s}  {'% ânc':>7s}  {'ROAS idx':>8s}")
            print("  " + "─" * 64)
            for item in rollup_df.columns:
                contrib_abs = float(rollup_df[item].sum())
                contrib_share = contrib_abs / (total + 1e-12)
                ss = float(spend_share_lookup.get(item, float("nan")))
                ri = contrib_share / ss if ss > 0 else float("nan")
                ri_str = f"{ri:.2f}" if ri == ri else "–"
                print(f"  {str(item):<42s} {contrib_abs:>12,.0f}  {contrib_abs/anchor:>7.1%}  {ri_str:>8s}")
            print(f"  {'TOTAL':<42s} {total:>12,.0f}  {total/anchor:>7.1%}")

In [ ]:
from batch import consolidate_results
from plots import analyze_batch

# vehicle_spec agora vem embutido em result.config.vehicle_spec — override não necessário
df_meta = consolidate_results(all_results)
print(f"Rollup levels por dim:")
print(df_meta.groupby(["dim", "rollup"])["item"].nunique().rename("n_items").to_string())

batch_figs = analyze_batch(all_results, df_meta, show=False)

for dim, fig in batch_figs.items():
    display(fig)

### Visualização em Árvore

Sunburst hierárquico por dimensão — tamanho do setor = share modelo, cor = ROAS Index.  
- **Ambiente**: `tipo → vertical → grupo → ambiente`  
- **Praça**: `estado → praça`  
- Dims sem hierarquia (Midia, Modelo de Planejamento) são puladas automaticamente.

In [ ]:
from plots import analyze_trees

# chart_type: "sunburst" | "treemap" | "icicle"
tree_figs = analyze_trees(all_results, chart_type="sunburst", show=False)

for key, fig in tree_figs.items():
    display(fig)

## 5. Exportar resultados

In [ ]:
import os

os.makedirs(OUTPUT_BASE_DIR, exist_ok=True)
csv_path = os.path.join(OUTPUT_BASE_DIR, "meta_analysis.csv")
df_meta.to_csv(csv_path, index=False)
print(f"CSV salvo: {csv_path}")

# Salvar figuras em HTML (interativo)
for dim, fig in batch_figs.items():
    html_path = os.path.join(OUTPUT_BASE_DIR, f"report_{dim.replace(' ', '_')}.html")
    fig.write_html(html_path)
    print(f"  {html_path}")